# Day 13: Evaluation — How Do You Know It Worked?

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> If you can't measure it, you can't improve it.

How do you evaluate an agent's output? Today we build an evaluation pipeline:
run the agent, check the output against criteria, and score it.

## What You'll Build
- A **customer-support agent** for a fictional SaaS product (CloudMail)
- An evaluation function that tests the 3 things that break in production:
  1. **Accuracy** — grounded in real facts (not invented)
  2. **Completeness** — answered every part of the question
  3. **Scope handling** — refuses politely when out-of-scope instead of hallucinating

The capital-of-France eval can't catch a hallucination. This one can.

In [1]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 13 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 13 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Evaluation Framework

For each answer, we score 3 criteria (1-5 each):
1. **Accuracy** — Is it grounded in the real product facts?
2. **Completeness** — Did it cover every part of the question?
3. **Scope handling** — Did it refuse politely when asked something it doesn't know?

Criterion #3 is the one production agents fail most. Hallucinating on
out-of-scope questions is the #1 agent failure mode — and a keyword-match
eval on trivia questions cannot detect it.

---
## Framework 1: OpenAI Agents SDK

In [2]:
from agents import Agent, Runner

model = get_openai_agents_model()

# ponytail: bake product facts into the system prompt — no RAG, no tools, one source of truth
PRODUCT_CONTEXT = """You are a support agent for CloudMail. Answer ONLY using these facts.
If a question is not covered by these facts, say: "I don't have that information."

Pricing: Starter (free), Pro $10/month, Team $50/month.
Limits: Starter 100 emails/day, Pro 10,000/day, Team 100,000/day.
Integrations: Slack, Salesforce, HubSpot, Zapier.
Support: email (all plans), priority chat (Pro), phone (Team)."""

support_agent = Agent(
    name="CloudMail Support",
    instructions=PRODUCT_CONTEXT,
    model=model,
)

# (question, expected_keywords, scope) — scope="refuse" means the correct answer is a refusal
questions = [
    ("How much does CloudMail Pro cost?", ["$10"], "answer"),
    ("What are the daily email limits for each plan?", ["100", "10,000", "100,000"], "answer"),
    ("Does CloudMail integrate with Notion?", [], "refuse"),       # Notion is NOT in the list
    ("What support channels does each plan include?", ["email", "chat", "phone"], "answer"),
    ("Can CloudMail send SMS messages?", [], "refuse"),            # out of scope
]

def eval_support(answer: str, expected_keywords: list, scope: str) -> dict:
    """Score accuracy, completeness, scope-handling. Catches hallucination on out-of-scope questions."""
    a = answer.lower()
    if scope == "refuse":
        refused = any(w in a for w in ["don't", "doesn't", "not", "no information", "unable"])
        # ponytail: refusing correctly = full marks; hallucinating an answer = fail
        return {"accuracy": 5 if refused else 1, "completeness": 5 if refused else 1, "conciseness": 5}
    hit = sum(1 for k in expected_keywords if k.lower() in a)
    return {
        "accuracy": 5 if hit == len(expected_keywords) else (3 if hit else 1),
        "completeness": 5 if hit == len(expected_keywords) else 2,
        "conciseness": 5 if len(answer) < 200 else 2,
    }

print("=" * 60)
print("OPENAI AGENTS SDK — EVALUATION")
print("=" * 60)

scores = []
for q, keywords, scope in questions:
    result = await Runner.run(support_agent, q)
    answer = result.final_output
    s = eval_support(answer, keywords, scope)
    avg = sum(s.values()) / 3
    scores.append(avg)
    print(f"\nQ: {q}")
    print(f"A: {answer[:120]}")
    print(f"Score: {s} → Avg: {avg:.1f}/5")

print(f"\n{'=' * 60}")
print(f"OVERALL: {sum(scores)/len(scores):.1f}/5 average across {len(questions)} questions")

OPENAI AGENTS SDK — EVALUATION

Q: How much does CloudMail Pro cost?
A: CloudMail Pro costs $10/month.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: What are the daily email limits for each plan?
A: The daily email limits for each plan are as follows:
- Starter plan: 100 emails/day
- Pro plan: 10,000 emails/day
- Team
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: Does CloudMail integrate with Notion?
A: I don't have that information.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: What support channels does each plan include?
A: Starter plan includes email support. Pro and Team both include email support and additional priority chat for Pro plan, 
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: Can CloudMail send SMS messages?
A: I don't have that information.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

OVERALL: 5.0/5 average across 5 que

---
## Framework 2: LangGraph

In [3]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

model = get_langgraph_model()

agent = create_react_agent(
    model,
    tools=[],
    prompt=PRODUCT_CONTEXT,
)

print("=" * 60)
print("LANGGRAPH — EVALUATION")
print("=" * 60)

scores = []
for q, keywords, scope in questions:
    result = agent.invoke({"messages": [HumanMessage(content=q)]})
    answer = result["messages"][-1].content
    s = eval_support(answer, keywords, scope)
    avg = sum(s.values()) / 3
    scores.append(avg)
    print(f"\nQ: {q}")
    print(f"A: {answer[:120]}")
    print(f"Score: {s} → Avg: {avg:.1f}/5")

print(f"\n{'=' * 60}")
print(f"OVERALL: {sum(scores)/len(scores):.1f}/5 average across {len(questions)} questions")

/var/folders/s0/2pw05fm942n5t1b5j7zzd_5m0000gn/T/ipykernel_77224/3013258122.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


LANGGRAPH — EVALUATION

Q: How much does CloudMail Pro cost?
A: CloudMail Pro costs $10/month.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: What are the daily email limits for each plan?
A: Starter has a limit of 100 emails per day, Pro has a limit of 10,000 emails per day, and Team has a limit of 100,000 ema
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: Does CloudMail integrate with Notion?
A: I don't have that information.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: What support channels does each plan include?
A: Starter plan includes email support. Pro plan has email and priority chat support. Team plan includes email, priority ch
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: Can CloudMail send SMS messages?
A: I don't have that information.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

OVERALL: 5.0/5 average across 5 questions


---
## Framework 3: CrewAI

In [5]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

# ponytail: backstory carries the product facts; goal tells it to refuse when out of scope
expert = Agent(
    role="CloudMail Support Agent",
    goal="Answer customer questions using ONLY the CloudMail facts. Politely refuse if not covered.",
    backstory=PRODUCT_CONTEXT,
    llm=llm,
    verbose=False,
)

print("=" * 60)
print("CREWAI — EVALUATION")
print("=" * 60)

scores = []
for q, keywords, scope in questions:
    task = Task(
        description=q,
        expected_output="A concise answer grounded in the CloudMail facts, or a polite refusal.",
        agent=expert,
    )
    crew = Crew(agents=[expert], tasks=[task], process=Process.sequential, verbose=False)
    # Jupyter runs an event loop (cell 4 uses top-level await), so CrewAI's sync kickoff()
    # refuses with RuntimeError "invoked synchronously from within a running event loop".
    # The async kickoff, awaited, is the fix that works here.
    result = await crew.kickoff_async()
    answer = str(result)
    s = eval_support(answer, keywords, scope)
    avg = sum(s.values()) / 3
    scores.append(avg)
    print(f"\nQ: {q}")
    print(f"A: {answer[:120]}")
    print(f"Score: {s} → Avg: {avg:.1f}/5")

print(f"\n{'=' * 60}")
print(f"OVERALL: {sum(scores)/len(scores):.1f}/5 average across {len(questions)} questions")

CREWAI — EVALUATION

Q: How much does CloudMail Pro cost?
A: CloudMail Pro costs $10 per month.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: What are the daily email limits for each plan?
A: For Starter plan, the daily email limit is 100 emails. For Pro plan, it's 10,000 emails/day. And for Team plan, the dail
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: Does CloudMail integrate with Notion?
A: I don't have that information.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

Q: What support channels does each plan include?
A: For CloudMail, the support plans are structured as follows:
- Starter plan includes email support only. There is no prio
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 2} → Avg: 4.0/5

Q: Can CloudMail send SMS messages?
A: I don't have that information.
Score: {'accuracy': 5, 'completeness': 5, 'conciseness': 5} → Avg: 5.0/5

OVERALL: 4.8/5 average across 5 questions


---
## Comparison: Evaluation

| Framework | Evaluation approach | Production eval tools |
|---|---|---|
| OpenAI SDK | Inspect `result.final_output` | OpenAI evals, custom scorers |
| LangGraph | Inspect message trace | LangSmith traces, graph replay |
| CrewAI | Inspect `kickoff()` result | Crew verbose output |

**Key insight:** Evaluation is framework-agnostic — you evaluate the OUTPUT, not the
framework. The framework just determines how you get the output and how much
trace information you have for debugging bad outputs.

LangGraph wins for debugging evaluations because the full message trace shows
exactly what the LLM was thinking at each step.

## Key Takeaway

Evaluation is framework-agnostic. What differs is **observability** — how much
information you have about WHY the agent gave a particular answer. LangGraph's
full message trace makes it the best for debugging agent quality.

The eval itself only matters if it tests the right things. A trivia eval
(capitals, math facts) cannot catch the #1 production failure mode: an agent
that hallucinates confidently on questions it has no answer for. Build eval
sets that include out-of-scope questions where the correct behavior is refusal.

---

